# AI Co-Scientist: Testing Hypotheses

* * * 

<div class="alert alert-success">  
    
### Learning Objectives 
    
* Compare competing hypotheses by held-out error and BIC.
* Estimate the uncertainty of fitted exponents with the bootstrap.
* Test whether a variable is needed in a law.
* Choose a resampling unit that matches how the data were collected.
* Interpret a blind result after the reveal.
</div>

### Icons Used in This Notebook
🔔 **Question**: A quick question to help you understand what's going on.<br>
🥊 **Challenge**: Interactive exercise. We'll work through these in the workshop!<br>
💡 **Tip**: How to do something a bit more efficiently or effectively.<br>
⚠️ **Warning:** Heads-up about tricky stuff or common mistakes.<br>

### Sections
1. [Competing Hypotheses](#compare)
2. [How Sure Are We?](#bootstrap)
3. [Does x2 Matter?](#mass)
4. [The Reveal](#reveal)

In [ ]:
# Imports and paths used in this notebook.
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'scripts' / '03_tests'))
import hypothesis_tests as ht
from evaluator import load_rows
from reveal import G_ACCEPTED, implied_g, load_key

In [ ]:
rows = load_rows(ROOT / 'data' / 'observations.csv')
train = [row for row in rows if row['split'] == 'train']
test = [row for row in rows if row['split'] == 'test']
train_logs = ht.logs(train)
test_logs = ht.logs(test)
all_logs = ht.logs(rows)

<a id='compare'></a>

# Competing Hypotheses

No language model in this part: everything is ordinary least squares on
`log10` values. Parts 1 and 2 left us with four hypotheses:

In [ ]:
for name, formula, free, fixed in ht.HYPOTHESES:
    print(name, formula)

Each one is fitted on the training groups and judged on the held-out groups.
We also report the **BIC** (Bayesian information criterion), which rewards a
good fit but charges a price for each free parameter. Lower is better for
every column.

In [ ]:
table = ht.compare_hypotheses(train_logs, test_logs)
print(f"{'':4}{'formula':32}{'params':>7}{'train':>8}{'held-out':>10}{'BIC':>9}")
for h in table:
    print(f"{h['name']:4}{h['formula']:32}{h['free_parameters']:7d}"
          f"{h['train_log_error']:8.3f}{h['test_log_error']:10.3f}{h['bic']:9.1f}")

💡 **Tip:** Errors are in **dex**, the mean error in `log10(y)`. An error of
0.036 dex means predictions are off by about 10^0.036 ≈ 1.09, or 9%, on
average.

🔔 **Question:** H3 and H4 tie on held-out error. H3 fits two exponents; H4
fixes them at 3/2 and −1/2. Which would you report, and why?

In [ ]:
(ROOT / 'results').mkdir(exist_ok=True)
ht.figure(rows, table, ROOT / 'results' / 'fig_pred_vs_obs.png')
Image(ROOT / 'results' / 'fig_pred_vs_obs.png')

## 🥊 Challenge 1: Add a Fifth Hypothesis

Test `y = c * x1^2 * x2^(-1)`. Its exponents are fixed, so the only free
parameter is `c`.

`ht.fit(free, fixed, logs)` fits a hypothesis. Pass `[]` for `free` (no free
exponents) and the exponents `(p, q)` as `fixed`. It returns
`(log10_c, p, q)`. `ht.residuals(params, logs)` returns the errors in
`log10(y)`. Report the mean absolute residual on the test rows.

In [ ]:
params = ht.fit(..., ..., train_logs)
test_error = ...
print(f'H5 held-out error: {test_error:.3f} dex')

<a id='bootstrap'></a>

# How Sure Are We?

H3 found exponents close to 3/2 and −1/2. How close is close enough? The
**bootstrap** answers this without any formulas:

1. Draw a new dataset of the same size, sampling with replacement.
2. Refit the law.
3. Repeat 5,000 times. The middle 95% of the refitted values is a 95% interval.

We resample in two ways: single rows, and whole groups.

In [ ]:
boot = ht.bootstrap(rows, np.random.default_rng(ht.SEED))
for how in ('rows', 'groups'):
    p_low, p_high = np.percentile(boot[how]['p'], [2.5, 97.5])
    q_low, q_high = np.percentile(boot[how]['q'], [2.5, 97.5])
    print(f'resampling {how:7}: p in [{p_low:.3f}, {p_high:.3f}]   q in [{q_low:.3f}, {q_high:.3f}]')

⚠️ **Warning:** All rows in a group share one measurement of `x2`. They are
not independent, so resampling single rows treats the data as more
informative than it is. Resample the unit that was measured independently:
here, the group.

## 🥊 Challenge 2: Plot the Bootstrap

Draw a histogram of the group-resampled `q` values, `boot['groups']['q']`,
with a vertical line at −0.5. Is −0.5 a plausible value for `q`?

💡 **Tip:** `ax.axvline(x)` draws a vertical line at `x`.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

# YOUR CODE HERE

plt.show()

<a id='mass'></a>

# Does x2 Matter?

H2 leaves out `x2`; H3 includes it. Two ways to test the **null hypothesis**
that `x2` does not matter:

- An **F-test** asks whether adding `x2` reduces the error more than chance would.
- A **group-level check** uses one data point per group, so the shared `x2`
  is only counted once.

In [ ]:
mt = ht.mass_tests(rows, all_logs)
f_test, group_level = mt['f_test'], mt['group_level']
print(f"F-test: F = {f_test['f']:.0f}, p = {f_test['p_value']:.1e}")
print(f"Group-level check ({group_level['n_groups']} groups): "
      f"q = {group_level['q']:.3f} ± {group_level['q_stderr']:.3f}, p = {group_level['p_value']:.1e}")

🔔 **Question:** Both tests say `x2` matters, but the group-level p-value is
20 orders of magnitude larger. Why is it the more honest of the two?

<a id='reveal'></a>

# The Reveal

Now, and only now, we open the key.

In [ ]:
key = load_key(ROOT / 'data' / 'key.json')
for name, meaning in key['meaning'].items():
    print(f'{name} = {meaning}')
print(key['groups'])

The data are real orbits of planets and moons. The law is **Kepler's third
law**, with the mass term that Newton added:

`T = 2π √(a³ / (G M))`

The period `T` grows as orbit size `a` to the power 3/2, and shrinks as the
central mass `M` to the power −1/2: exactly H4. Undo the unit changes and the
fitted constant gives the gravitational constant `G`.

In [ ]:
log_c_h4 = ht.fit([], (1.5, -0.5), all_logs)[0]
g_estimate = implied_g(10 ** log_c_h4, key)
print(f'G = {g_estimate:.3e} (accepted {G_ACCEPTED:.3e}, {100 * (g_estimate / G_ACCEPTED - 1):+.1f}%)')

## 🥊 Challenge 3: Which Interval Contains G?

`boot[how]['log10_c_h4']` holds the bootstrap values of `log10(c)` for H4.
For each way of resampling, turn the 2.5th and 97.5th percentiles into values
of `G` with `implied_g(10 ** value, key)`, and check whether the interval
contains `G_ACCEPTED`.

⚠️ **Warning:** `implied_g()` reverses the order: a larger `c` gives a smaller
`G`. Use `sorted()` on the two ends.

In [ ]:
for how in ('rows', 'groups'):
    # YOUR CODE HERE
    ...

The interval from resampling rows misses the accepted value of `G`. The
interval from resampling groups contains it. Same data, same law: only the
honesty of the uncertainty changed.

Finally, run the full analysis script. It saves everything to `results/`,
which Part 4 needs.

In [ ]:
!{sys.executable} ../scripts/03_tests/hypothesis_tests.py

<div class="alert alert-success">

## ❗ Key Points

* Judge competing hypotheses on held-out groups, and prefer the simpler law when they tie.
* The bootstrap gives uncertainty by refitting on resampled data.
* Resample the unit that was measured independently: here, whole groups.
* Test whether a variable matters, and count shared measurements only once.
* The blind result recovers Kepler's third law with Newton's mass term, and G within 9%.

</div>